# FragilityShield — Universal Quantitative Research Lab & Test All Orchestrator
### Complete Google Colab Integration Notebook

This notebook connects your Google Colab session directly to your running **FragilityShield** instance to perform automated, end-to-end model validation, portfolio risk analysis, macro stress testing, tail-risk EVT analysis, and fragility matrix tests.

#### What This Notebook Tests:
1. **Zero Installation Required:** 100% self-contained pure-Python client (uses only `urllib`, `json`, and `numpy`). No Git repository cloning or package building needed.
2. **Pre-flight Compatibility Validation:** Validates input dimensions against 9 quantitative testing engines before execution.
3. **"Test All" Quantitative Suite:** Executes all 9 institutional test engines simultaneously:
   - **Data Diagnostics:** ADF stationarity, Ljung-Box autocorrelation, Jarque-Bera normality.
   - **Risk Analytics:** Parametric & Historical VaR (95%/99%), Expected Shortfall (CVaR), Volatility, Sharpe, Max Drawdown.
   - **Macro Stress Lab:** COVID-19 crash (-28%), Lehman Brothers 2008 (-35%), Volatility Spike (+150%).
   - **Tail Risk & EVT Reverse Stress:** Extreme Value Theory Hill index and distance in $\sigma$ to capital breach.
   - **Fragility Assumption Matrix:** Sensitivity to execution slippage (0-25 bps) and friction costs.
   - **Monte Carlo Simulation:** 1,000 geometric Brownian motion paths, terminal quantiles, ruin probability.
   - **Stationary Bootstrap:** 500 block resamples with 95% confidence intervals.
   - **Model Validation:** Kolmogorov-Smirnov Goodness-of-Fit and Kupiec POF test.
   - **Strategy Backtesting:** Honest applicability checks without metric fabrication.
4. **Custom Test Profiles:** Select individual modules and apply custom parameter overrides.
5. **Declarative Configurations:** Save and persist custom strategies and scenarios (zero `eval`/`exec`).
6. **Artifact Upload with SHA-256 Checksums:** Cryptographic model checkpoint storage.
7. **Unified Dashboard Ledger:** Immediate propagation to the FragilityShield Experiment History and Command Center.


## Step 0: Standalone FragilityShield REST Client (Zero-Install)

Run this cell to initialize the FragilityShield REST Client. It works completely out-of-the-box in Google Colab using only Python standard library.

In [ ]:
import sys
import os
import json
import time
import hashlib
import urllib.request
import urllib.error
import numpy as np

class ExternalMetric:
    def __init__(self, metric_name, observed_value, unit="", sample_count=None, is_externally_reported=True):
        self.metric_name = metric_name
        self.observed_value = observed_value
        self.unit = unit
        self.sample_count = sample_count
        self.is_externally_reported = is_externally_reported

    def to_dict(self):
        return {
            "metric_name": self.metric_name,
            "observed_value": self.observed_value,
            "unit": self.unit,
            "sample_count": self.sample_count,
            "is_externally_reported": self.is_externally_reported,
        }

class ExternalPrediction:
    def __init__(self, timestamp, target, predicted_value, actual_value=None, horizon="1D", confidence_lower=None, confidence_upper=None):
        self.timestamp = timestamp
        self.target = target
        self.predicted_value = predicted_value
        self.actual_value = actual_value
        self.horizon = horizon
        self.confidence_lower = confidence_lower
        self.confidence_upper = confidence_upper

    def to_dict(self):
        return {
            "timestamp": self.timestamp,
            "target": self.target,
            "predicted_value": self.predicted_value,
            "actual_value": self.actual_value,
            "horizon": self.horizon,
            "confidence_lower": self.confidence_lower,
            "confidence_upper": self.confidence_upper,
        }

class StandaloneOrchestratorClient:
    def __init__(self, api_url, api_token="", workspace_id="default_workspace"):
        self.api_url = api_url.rstrip("/")
        self.api_token = api_token
        self.workspace_id = workspace_id

    def _headers(self):
        h = {"Content-Type": "application/json", "Bypass-Tunnel-Reminder": "true", "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        if self.api_token:
            h["Authorization"] = f"Bearer {self.api_token}"
        return h

    def list_modules(self):
        req = urllib.request.Request(f"{self.api_url}/api/orchestrator/modules", headers=self._headers())
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8")).get("modules", [])

    def validate(self, experiment_type="model", data=None, profile_id="test_all", selected_modules=None):
        payload = {
            "experiment_type": experiment_type,
            "data": data or {},
            "profile_id": profile_id,
            "selected_modules": selected_modules,
        }
        data_bytes = json.dumps(payload).encode("utf-8")
        req = urllib.request.Request(f"{self.api_url}/api/orchestrator/validate", data=data_bytes, headers=self._headers(), method="POST")
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8"))

    def test_all(self, experiment_id, run_id=None, experiment_type="model", data=None, display_name=None, assumptions=None):
        payload = {
            "experiment_id": experiment_id,
            "run_id": run_id or f"run_colab_{int(time.time())}",
            "display_name": display_name or f"Colab Test All Suite {experiment_id}",
            "submission_source": "google_colab",
            "experiment_type": experiment_type,
            "profile_id": "test_all",
            "data": data or {},
            "config_overrides": {"assumptions": assumptions or {}} if assumptions else {},
        }
        data_bytes = json.dumps(payload).encode("utf-8")
        req = urllib.request.Request(f"{self.api_url}/api/orchestrator/run", data=data_bytes, headers=self._headers(), method="POST")
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8"))

    def run_profile(self, experiment_id, profile_id="custom", experiment_type="model", selected_modules=None, data=None, config_overrides=None, display_name=None):
        payload = {
            "experiment_id": experiment_id,
            "run_id": f"run_colab_profile_{int(time.time())}",
            "display_name": display_name or f"Colab Profile {profile_id}",
            "submission_source": "google_colab",
            "experiment_type": experiment_type,
            "profile_id": profile_id,
            "selected_modules": selected_modules,
            "data": data or {},
            "config_overrides": config_overrides or {},
        }
        data_bytes = json.dumps(payload).encode("utf-8")
        req = urllib.request.Request(f"{self.api_url}/api/orchestrator/run", data=data_bytes, headers=self._headers(), method="POST")
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8"))

    def get_run(self, run_id):
        req = urllib.request.Request(f"{self.api_url}/api/orchestrator/runs/{run_id}", headers=self._headers())
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8"))

    def save_strategy(self, strategy_config):
        data_bytes = json.dumps(strategy_config).encode("utf-8")
        req = urllib.request.Request(f"{self.api_url}/api/configurations/strategies", data=data_bytes, headers=self._headers(), method="POST")
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8"))

    def save_scenario(self, scenario_config):
        data_bytes = json.dumps(scenario_config).encode("utf-8")
        req = urllib.request.Request(f"{self.api_url}/api/configurations/scenarios", data=data_bytes, headers=self._headers(), method="POST")
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8"))

class StandaloneExternalClient:
    def __init__(self, api_url, api_token="", workspace_id="default_workspace"):
        self.api_url = api_url.rstrip("/")
        self.api_token = api_token
        self.workspace_id = workspace_id

    def register_model(self, model_id, name, version="1.0.0", author="colab", description=""):
        return {"model_id": model_id, "name": name, "version": version}

    def register_dataset(self, dataset_id, name, version="1.0.0", format="csv", symbols=()):
        return {"dataset_id": dataset_id, "name": name, "symbols": symbols}

    def submit_run(self, experiment_id, run_id, model_id, target_lab="general", metrics=None, predictions=None, parameters=None, run_status="completed", dataset_id=None, environment=None, auto_test_all=False, returns=None):
        payload = {
            "schema_version": "1.0.0",
            "workspace_id": self.workspace_id,
            "experiment_id": experiment_id,
            "run_id": run_id,
            "model_id": model_id,
            "dataset_id": dataset_id,
            "target_lab": target_lab,
            "run_status": run_status,
            "parameters": parameters or {},
            "metrics": [m.to_dict() if hasattr(m, "to_dict") else m for m in (metrics or [])],
            "predictions": [p.to_dict() if hasattr(p, "to_dict") else p for p in (predictions or [])],
            "environment": environment or {"environment_type": "google_colab"},
            "auto_test_all": auto_test_all,
            "returns": returns or [],
        }
        data = json.dumps(payload).encode("utf-8")
        headers = {"Content-Type": "application/json", "Bypass-Tunnel-Reminder": "true", "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        if self.api_token:
            headers["Authorization"] = f"Bearer {self.api_token}"
        req = urllib.request.Request(f"{self.api_url}/api/ingest/runs", data=data, headers=headers, method="POST")
        with urllib.request.urlopen(req) as resp:
            res_dict = json.loads(resp.read().decode("utf-8"))
            class ResEnvelope:
                def __init__(self, d):
                    self.__dict__.update(d)
            return ResEnvelope(res_dict)

    def upload_artifact(self, run_id, file_path, artifact_id, name, expected_sha256=None):
        content = open(file_path, "rb").read()
        url = f"{self.api_url}/api/ingest/artifacts?run_id={run_id}&artifact_id={artifact_id}&name={name}"
        if expected_sha256:
            url += f"&checksum_sha256={expected_sha256}"
        headers = {"Content-Type": "application/octet-stream", "Bypass-Tunnel-Reminder": "true", "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        if self.api_token:
            headers["Authorization"] = f"Bearer {self.api_token}"
        req = urllib.request.Request(url, data=content, headers=headers, method="POST")
        with urllib.request.urlopen(req) as resp:
            res = json.loads(resp.read().decode("utf-8"))
            class ArtEnvelope:
                def __init__(self, d):
                    self.__dict__.update(d)
            return ArtEnvelope(res.get("artifact", {}))

    def get_run(self, run_id):
        headers = {"Bypass-Tunnel-Reminder": "true", "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        if self.api_token:
            headers["Authorization"] = f"Bearer {self.api_token}"
        req = urllib.request.Request(f"{self.api_url}/api/ingest/runs/{run_id}", headers=headers)
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8")).get("run")

class Client:
    def __init__(self, api_url, api_token="", workspace_id="default_workspace", environment_type="google_colab"):
        self.api_url = api_url.rstrip("/")
        self.api_token = api_token
        self.workspace_id = workspace_id
        self.external = StandaloneExternalClient(api_url=api_url, api_token=api_token, workspace_id=workspace_id)
        self.orchestrator = StandaloneOrchestratorClient(api_url=api_url, api_token=api_token, workspace_id=workspace_id)

print("✓ FragilityShield Colab Client ready (Zero-Install: External + Orchestrator).")


## Step 1: Environment Configuration & Endpoint Setup

Set your target FragilityShield URL (e.g. `https://fragshield-dev.loca.lt` or local network URL).

In [ ]:
# Step 1: Endpoint & Connection Verification
try:
    from google.colab import userdata
    api_token = userdata.get("FRAGILITYSHIELD_API_TOKEN")
except (ImportError, Exception):
    api_token = os.environ.get("FRAGILITYSHIELD_API_TOKEN", "dev_test_bearer_token")

# Active Cloudflare Tunnel (No password / No bypass prompt required)
api_url = os.environ.get("FRAGILITYSHIELD_API_URL", "https://approve-declined-missions-eden.trycloudflare.com")

print(f"Target Endpoint: {api_url}")
print(f"API Token configured: {bool(api_token)}")

# Verify immediate connectivity
try:
    req = urllib.request.Request(f"{api_url}/api/health", headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        health = json.loads(resp.read().decode("utf-8"))
        print(f"\n✓ Successfully connected to: {health.get('engine')}!")
        print(f"  Status: {health.get('status')} | Version: {health.get('version')}")
except Exception as e:
    print(f"\n❌ Connection Error: {e}")
    print("\nTroubleshooting Tips:")
    print("1. Do NOT use http://127.0.0.1:8000 inside Google Colab (that refers to Google's VM).")
    print("2. Use the Cloudflare Tunnel: https://approve-declined-missions-eden.trycloudflare.com")
    print("3. Ensure the backend server and tunnel are active on the host machine.")

client = Client(
    api_url=api_url,
    api_token=api_token,
    workspace_id="ws_colab_alpha_research",
    environment_type="google_colab"
)
print("\n✓ Client initialized.")


## Step 2: Discover Registered Quantitative Engines & Pre-Flight Validation

Query available test modules and run pre-flight compatibility checks before executing tests.

In [ ]:
modules = client.orchestrator.list_modules()
print(f"Discovered {len(modules)} Institutional Testing Engines:")
for m in modules:
    print(f"  • [{m["module_id"]}] {m["name"]}")

# Pre-flight check with sample input
val = client.orchestrator.validate(
    experiment_type="model",
    data={"returns": [0.01, -0.02, 0.005, -0.012, 0.008]},
    profile_id="test_all"
)
print(f"\nPre-Flight Compatibility: {val.get("applicable_count")}/{val.get("total_modules")} engines ready.")


## Step 3: Generate Realistic Model Predictions & Return Series

Generate 50 realistic intraday return steps with realistic statistical properties (mean return, volatility, heavy tails).

In [ ]:
rng = np.random.RandomState(42)
n_steps = 60

# Realistic returns series (Student-t with mild negative skew)
returns_data = (rng.standard_t(df=6, size=n_steps) * 0.012 - 0.0005).tolist()
returns_data = [round(float(r), 6) for r in returns_data]

actuals = np.array(returns_data[:30])
forecasts = actuals + rng.normal(0.0, 0.002, size=len(actuals))

predictions = [
    ExternalPrediction(
        timestamp=f"2026-10-{i+1:02d}T15:30:00+05:30",
        target="NIFTY_RETURN",
        predicted_value=round(float(forecasts[i]), 6),
        actual_value=round(float(actuals[i]), 6),
        horizon="1D"
    )
    for i in range(len(actuals))
]

print(f"Generated {len(returns_data)} return observations and {len(predictions)} validation predictions.")


## Step 4: Execute "Test All" Quantitative Orchestrator Suite

Trigger the Shared Test Orchestrator across all 9 engines simultaneously. Inspect the passed, failed, and unsupported tests.

In [ ]:
suite_run = client.orchestrator.test_all(
    experiment_id="exp_colab_alpha_01",
    run_id=f"colab_suite_{int(time.time())}",
    display_name="Colab Full Quantitative Robustness Suite",
    experiment_type="model",
    data={
        "returns": returns_data,
        "assumptions": {
            "slippage_bps": 5.0,
            "cost_bps": 10.0
        }
    }
)

print(f"\nTest All Suite Results:")
print(f"  • Status:           {suite_run.get('status')}")
print(f"  • Run ID:           {suite_run.get('run_id')}")
print(f"  • Summary:          {suite_run.get('summary_metrics')}")

results = suite_run.get("test_results", {})
print(f"\nIndividual Module Execution Statuses:")
for mod_id, res in results.items():
    status_badge = res.get('status', '').upper()
    summary = res.get('summary_metric', '')
    print(f"  [{status_badge:11s}] {mod_id:30s} -> {summary}")


## Step 5: Execute Custom Test Profile with Parameter Overrides

Run targeted stress testing and Monte Carlo simulations with custom parameters (e.g. 2,000 paths, 180 days).

In [ ]:
custom_run = client.orchestrator.run_profile(
    experiment_id="exp_colab_stress",
    profile_id="custom",
    experiment_type="model",
    display_name="Colab Custom Monte Carlo & Stress Profile",
    selected_modules=["risk_analytics", "scenario_stress", "monte_carlo_simulation"],
    data={"returns": returns_data},
    config_overrides={
        "monte_carlo_simulation": {
            "n_paths": 2000,
            "n_days": 180
        }
    }
)

print(f"Custom Profile Status: {custom_run.get('status')}")
mc_metrics = custom_run.get("test_results", {}).get("monte_carlo_simulation", {}).get("metrics", {})
print(f"Monte Carlo 2000-Path Summary:")
for k, v in mc_metrics.items():
    print(f"  • {k}: {v.get('observed_value')}")


## Step 6: Declarative Custom Strategy & Scenario Registration (Zero eval/exec)

Register institutional strategies and stress scenarios directly via declarative JSON schemas.

In [ ]:
# 1. Save Custom Strategy Specification
strategy_spec = {
    "strategy_id": "colab_vol_breakout_v1",
    "name": "Colab Volatility Breakout System",
    "description": "Intraday volatility breakout with dynamic ATR sizing.",
    "version": "1.0.0",
    "author": "colab_researcher",
    "parameters": {
        "atr_period": 14,
        "multiplier": 2.0,
        "holding_period": 5
    },
    "cost_model": {
        "slippage_bps": 5.0,
        "commission_bps": 2.0
    },
    "risk_limits": {
        "max_drawdown_pct": 15.0
    }
}
strat_res = client.orchestrator.save_strategy(strategy_spec)
print(f"Custom Strategy Registered: {strat_res.get('strategy', {}).get('strategy_id', 'strat_saved')} (v{strat_res.get('strategy', {}).get('version', '1.0.0')})")

# 2. Save Custom Macro Scenario
scenario_spec = {
    "scenario_id": "colab_sovereign_downgrade",
    "name": "Sovereign Credit Downgrade Shock",
    "description": "Simulated 400bps yield spike with 8.5% equity decline.",
    "version": "1.0.0",
    "shock_type": "macro",
    "shocks": {
        "equity_shock": -0.085,
        "yield_shock_bps": 400.0,
        "inr_depreciation": 0.045
    }
}
scen_res = client.orchestrator.save_scenario(scenario_spec)
print(f"Custom Scenario Registered: {scen_res.get('scenario', {}).get('scenario_id', 'scen_saved')}")


## Step 7: Cryptographic Artifact Upload with SHA-256 Checksum

Save and upload model checkpoints or configuration weights with tamper-evident cryptographic validation.

In [ ]:
checkpoint_data = {
    "weights": [0.05, 0.85, 0.10],
    "garch_alpha": 0.08,
    "garch_beta": 0.88,
    "trained_in": "google_colab"
}
local_file = "model_weights.json"
with open(local_file, "w") as f:
    json.dump(checkpoint_data, f)

file_sha = hashlib.sha256(open(local_file, "rb").read()).hexdigest()

artifact_ref = client.external.upload_artifact(
    run_id=suite_run.get("run_id"),
    file_path=local_file,
    artifact_id="art_garch_weights_v1",
    name="garch_weights.json",
    expected_sha256=file_sha,
)

print(f"Uploaded artifact: {artifact_ref.name}")
print(f"SHA-256 Verified: {artifact_ref.checksum_sha256}")


## Step 8: Verify Dashboard & Central Ledger Integration

Retrieve the finalized run record from the FragilityShield Experiment Registry. Open your browser dashboard to view the charts and diagnostics live!

In [ ]:
run_details = client.orchestrator.get_run(suite_run.get("run_id"))

print("Central Platform Execution Ledger Record:")
print(f"  • Display Name:    {run_details.get('display_name')}")
print(f"  • Run Status:      {run_details.get('status')}")
print(f"  • Provenance Hash: {run_details.get('provenance_hash')}")
print(f"  • Summary Metrics: {run_details.get('summary_metrics')}")
print("\n✓ Verified! You can now view this run on the FragilityShield Dashboard at /experiments.")
